# Antibody lineage pre-filtering benchmark

SymScan on a short CDR3 k-mer as a candidate generator for clonal lineage
identification, against exhaustive pairwise Hamming comparison within each
V/J/length group.

Writes `../data/antibody_benchmark_results.json`; plotted by `pub/antibody_lineages.ipynb`.

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import pyrepseq as prs
import pwseqdist
import symscan
from tqdm.auto import tqdm

import benchutils as bu

DATADIR = '../data/'

# minimum number of sequences per VJl group
min_seq_count = 100
# proportional cdr3 nt length cutoff for identifying putative clonal lineages
prop_length_cutoff = 0.15
# length of the CDR3 k-mer handed to SymScan as the candidate generator
k = 6
max_editss = [1, 2, 3]

In [ ]:
bu.describe_env()

In [ ]:
df = pd.read_csv(f'{DATADIR}/briney_316188.tsv.gz',
                usecols=['sequence_id', 'v_call', 'j_call', 'junction'], sep='\t', index_col=0)
df.rename(columns={'v_call': 'v_gene', 'j_call': 'j_gene', 'junction': 'cdr3_nt'}, inplace=True)
df.drop_duplicates(subset=['v_gene', 'j_gene', 'cdr3_nt'], inplace=True)
df['cdr3_length'] = df['cdr3_nt'].str.len()

In [ ]:
qlow, qhigh = df['cdr3_length'].quantile(1e-3), df['cdr3_length'].quantile(1-1e-3)
print(f'Filtering outlier cdr3 lengths outside of [{qlow}, {qhigh}]')
mask = (df['cdr3_length'] > qlow) & (df['cdr3_length'] < qhigh)
df = df[mask]
print('Mean length cutoff: %f' % (np.mean(df['cdr3_length'])*prop_length_cutoff))
print('Number of sequences: %e' % len(df))

In [ ]:
vjls = (df
        .value_counts(['v_gene', 'j_gene', 'cdr3_length'])
        .to_frame('sequence_count')
        .reset_index()
    )
topn = vjls[vjls['sequence_count'] > min_seq_count].shape[0]
topn

In [ ]:
dfs = []
for i in range(topn):
    v, j, l = vjls.iloc[i][['v_gene', 'j_gene', 'cdr3_length']]
    dfs.append(df[(df['v_gene'] == v) & (df['j_gene'] == j) & (df['cdr3_length'] == l)]['cdr3_nt'])
min([len(d) for d in dfs]), max([len(d) for d in dfs])

In [ ]:
def symscan_neighbors(sample, k, max_edits):
    l = len(sample.iloc[0])
    start = int(l//2-k//2)
    end = start + k
    seqs = list(sample.str.slice(start, end))
    (row, col, dists) = symscan.get_neighbors_within(seqs, max_distance=max_edits, distance_type='hamming')
    edges = np.column_stack((row, col))
    filtered = pwseqdist.apply_pairwise_sparse(metric=pwseqdist.metrics.nb_vector_hamming_distance,
                            seqs=np.asarray(sample, dtype=str), pairs=edges, use_numba=True)
    return filtered

In [ ]:
times_symscan = {}
ptotal_symscan = {}
for max_edits in max_editss:
    ptotal_symscan[max_edits] = 0
    times_symscan[max_edits] = 0
time_exhaustive = 0
ptotal_exhaustive = 0
for sample in tqdm(dfs):
    l = len(sample.iloc[0])
    told = time.time()
    pdists = prs.squareform(pwseqdist.apply_pairwise_rect(metric=pwseqdist.metrics.nb_vector_hamming_distance,
                        seqs1=np.asarray(sample, dtype=str), seqs2=None, use_numba=True))
    ptotal_exhaustive += np.sum(pdists<l*prop_length_cutoff)
    time_exhaustive += time.time()-told

    for max_edits in max_editss:
        told = time.time()
        pdists_symscan = symscan_neighbors(sample, k=k, max_edits=max_edits)
        ptotal_symscan[max_edits] += np.sum(pdists_symscan<l*prop_length_cutoff)
        times_symscan[max_edits] += time.time()-told
fractions = {k_: ptotal_symscan[k_]/ptotal_exhaustive for k_ in ptotal_symscan}
[fractions[i] for i in max_editss], time_exhaustive, times_symscan

In [ ]:
results = {
    'times_symscan': times_symscan,
    'time_exhaustive': time_exhaustive,
    'fractions': fractions,
    'max_editss': max_editss,
}
with open(f'{DATADIR}/antibody_benchmark_results.json', 'w') as f:
    json.dump(results, f)